# Feature Engineering Pipeline with Hyperparameter Tuning for Vertex AI

## 🎯 For Stakeholders: Simple Usage Guide

**You only need to do ONE thing: Edit `config.yaml` with your settings!**

This notebook automatically:
1. ✅ Loads your `config.yaml` file
2. ✅ Builds the training package as tar.gz
3. ✅ Uploads it to GCS automatically
4. ✅ Converts all config values to pipeline arguments
5. ✅ Creates and runs a Vertex AI Custom Training Job
6. ✅ **Sequential Execution:** Feature Engineering → Hyperparameter Tuning

### Quick Start:
1. **Edit `config.yaml`** - Update your project settings, SQL queries, model parameters, etc.
2. **Run all cells** - Execute the notebook cells in order (Cell → Run All)
3. **Done!** - The pipeline will run automatically with your configuration

### What you can configure in `config.yaml`:
- GCP project settings (project, dataset, bucket, etc.)
- BigQuery SQL queries for training/test data
- Data processing parameters (target column, variance threshold, etc.)
- Feature engineering settings (one-hot encoding, feature classification)
- Model configuration (XGBoost parameters)
- Output settings (file prefixes, result locations, **table names**)
- Hyperparameter tuning settings (trials, parameter ranges, input tables)
- Component enable/disable flags

### Pipeline Flow:
- **Feature Engineering** (if enabled) → Creates selected feature tables in BigQuery with fixed names
- **Hyperparameter Tuning** (if enabled) → Uses specified tables or FE output to find optimal model parameters
- **Sequential Execution:** When both are enabled, HP runs AFTER FE completes

### Key Features:
- ✅ **Fixed table names** (no timestamps) for easy reuse
- ✅ **Table reuse support** - Skip FE and directly use existing tables for HP tuning
- ✅ **Flexible configuration** - Enable/disable components as needed

**Note:** All configuration is automatically passed to the training job - no need to upload config.yaml separately!


## Step 1: Import Libraries and Helper Functions


In [1]:
# Import necessary libraries
import os
import json
import base64
from datetime import datetime
from google.cloud import aiplatform
from google.cloud import storage
from kfp import compiler

# Import helper functions (handles package building, uploading, etc.)
from pipeline_helpers import (
    build_and_upload_package,
    create_custom_training_job_component,
    create_hyperparameter_tuning_component
)

# Import config loader
from trainer.config import load_config

# ✅ CRITICAL: Import the corrected pipeline definition
# This ensures SEQUENTIAL execution (FE → HP), not conditional
from new_pipeline_definition import flexible_ml_pipeline

print("✅ All imports successful")
print("✅ Pipeline definition imported from new_pipeline_definition.py")
print("   → Sequential execution: Feature Engineering → Hyperparameter Tuning")
print("   → Fixed table names (no timestamps)")
print("   → Table reuse support")


✅ Feature Engineering component created
   Output tables will be: {{channel:task=;name=gcp_gcp_project;type=String;}}.{{channel:task=;name=gcp_db;type=String;}}.{{channel:task=;name=prefix;type=String;}}_selected_features_train
                          {{channel:task=;name=gcp_gcp_project;type=String;}}.{{channel:task=;name=gcp_db;type=String;}}.{{channel:task=;name=prefix;type=String;}}_selected_features_test
✅ Using Python package module: trainer.hp_tuning.task
✅ Hyperparameter tuning will use location: None
✅ Pipeline configured for SEQUENTIAL execution:
   1️⃣  Feature Engineering (creates tables)
   2️⃣  Hyperparameter Tuning (uses those tables)

🎯 Pipeline Mode: FULL PIPELINE (FE → HP)
✅ All imports successful
✅ Pipeline definition imported from new_pipeline_definition.py
   → Sequential execution: Feature Engineering → Hyperparameter Tuning
   → Fixed table names (no timestamps)
   → Table reuse support


## Step 2: Load Configuration from config.yaml


In [2]:
# Load your configuration from config.yaml
config = load_config("config.yaml")

# Setup pipeline constants (automatically extracted from config.yaml)
constants = {
    "PROJECT": config['gcp']['project'],
    "COMPUTE_PROJECT": config['gcp']['project'],
    "LOCATION": config['vertex_ai']['location'],
    "CMEK_KEY": config['vertex_ai']['cmek_key'],
    "SERVICE_ACCOUNT": config['vertex_ai']['service_account'],
    "DOCKER_URI": config['vertex_ai']['docker_uri'],
    "LOB": config['bigquery_labels']['lob'],
    "SHARED_PROJECT": config['gcp']['gcp_project'],
    "LABELS": {
        'owner': config['bigquery_labels']['owner'],
        'costcenter': config['bigquery_labels']['costcenter'],
        'tenant': 'hcm-cm-de',
        'self_serve': 'true',
        'lob': config['bigquery_labels']['lob'],
        'pipeline_type': config['bigquery_labels']['pipeline_type']
    }
}

# Get component enable flags
components_config = config.get('components', {})
enable_fe = components_config.get('feature_engineering', {}).get('enable', True)
enable_hp = components_config.get('hyperparameter_tuning', {}).get('enable', True)

print("✅ Configuration loaded from config.yaml")
print(f"   📍 Vertex AI Region: {constants['LOCATION']}")
print(f"   🐳 Docker Image: {constants['DOCKER_URI']}")
print(f"\n✅ Component Configuration:")
print(f"   Feature Engineering: {'ENABLED' if enable_fe else 'DISABLED'}")
print(f"   Hyperparameter Tuning: {'ENABLED' if enable_hp else 'DISABLED'}")
if enable_fe and enable_hp:
    print(f"   🔄 Execution Mode: SEQUENTIAL (FE → HP)")


c:\Users\A974930\Downloads\Github Action Test\vertex-pipeline-demo\src\Training\feature-eng\fe-src\test\trainer\config.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Loading config from: config.yaml
✅ Configuration loaded from config.yaml
   📍 Vertex AI Region: us-east4
   🐳 Docker Image: us-docker.pkg.dev/vertex-ai/training/xgboost-cpu.2-1:latest

✅ Component Configuration:
   Feature Engineering: DISABLED
   Hyperparameter Tuning: ENABLED


## Step 3: Build and Upload Training Package


In [3]:
# Get pipeline root from config
pipeline_root = config['vertex_ai']['pipeline_root']

# Build and upload package
package_uri, package_file = build_and_upload_package(
    training_dir=".",
    output_dir="dist",
    project=constants["PROJECT"],
    pipeline_root=pipeline_root
)

print(f"\n✅ Package built and uploaded: {package_uri}")


Building package from ....
STDOUT: running sdist
running egg_info
writing feature_engineering_trainer.egg-info\PKG-INFO
writing dependency_links to feature_engineering_trainer.egg-info\dependency_links.txt
writing requirements to feature_engineering_trainer.egg-info\requires.txt
writing top-level names to feature_engineering_trainer.egg-info\top_level.txt
reading manifest file 'feature_engineering_trainer.egg-info\SOURCES.txt'
writing manifest file 'feature_engineering_trainer.egg-info\SOURCES.txt'
running check
creating feature_engineering_trainer-0.1.0
creating feature_engineering_trainer-0.1.0\feature_engineering_trainer.egg-info
creating feature_engineering_trainer-0.1.0\trainer
creating feature_engineering_trainer-0.1.0\trainer\hp_tuning
creating feature_engineering_trainer-0.1.0\utils
copying files to feature_engineering_trainer-0.1.0...
copying config.yaml -> feature_engineering_trainer-0.1.0
copying setup.py -> feature_engineering_trainer-0.1.0
copying feature_engineering_train

## Step 4: Compile Pipeline Definition


In [4]:
# Compile the pipeline
compiler.Compiler().compile(
    pipeline_func=flexible_ml_pipeline,
    package_path="flexible_ml_pipeline.json"
)

print("✅ Pipeline compiled successfully")
print("   📄 Output: flexible_ml_pipeline.json")

# Display component status
hp_config = config.get('hyperparameter_tuning', {})
print(f"\n✅ Component Status:")
print(f"   Feature Engineering: {'ENABLED' if enable_fe else 'DISABLED'}")
if enable_hp:
    print(f"   Hyperparameter Tuning: ENABLED")
    print(f"      Max trials: {hp_config.get('max_trial_count', 10)}")
    print(f"      Parallel trials: {hp_config.get('parallel_trial_count', 2)}")
    print(f"      Method: {'Package Module (trainer.hp_tuning.task)' if hp_config.get('use_package_module', True) else 'GCS File (legacy)'}")
else:
    print(f"   Hyperparameter Tuning: DISABLED")


✅ Pipeline compiled successfully
   📄 Output: flexible_ml_pipeline.json

✅ Component Status:
   Feature Engineering: DISABLED
   Hyperparameter Tuning: ENABLED
      Max trials: 10
      Parallel trials: 2
      Method: Package Module (trainer.hp_tuning.task)


## Step 5: Prepare Pipeline Parameters


In [5]:
# Helper function to encode JSON arguments as base64
def encode_json_arg(value):
    """Encode a JSON-serializable value as base64 string."""
    json_str = json.dumps(value, ensure_ascii=False, separators=(',', ':'))
    encoded = base64.b64encode(json_str.encode('utf-8')).decode('utf-8')
    return f'base64:{encoded}'

# Encode all JSON config values
sql_queries_json = encode_json_arg(config['sql_queries'])
feature_classification_json = encode_json_arg(config['feature_classification'])
model_config_json = encode_json_arg(config['model'])
undersampling_config_json = encode_json_arg(config['undersampling'])
rfecv_config_json = encode_json_arg(config['rfecv'])
metrics_config_json = encode_json_arg(config['metrics'])
output_config_json = encode_json_arg(config['output'])
bigquery_query_config_json = encode_json_arg(config['bigquery_query'])
exclude_for_variance_json = encode_json_arg(config['data_processing']['exclude_for_variance'])
exclude_for_classification_json = encode_json_arg(config['data_processing']['exclude_for_classification'])

# Handle accelerator type (convert empty/null to None)
accelerator_type_value = config['vertex_ai'].get('accelerator_type', None)
if isinstance(accelerator_type_value, str):
    accelerator_type_value = accelerator_type_value.strip()
    if not accelerator_type_value or accelerator_type_value.lower() in ("null", "none", ""):
        accelerator_type_value = None

# Hyperparameter tuning parameters
hp_config = config.get('hyperparameter_tuning', {})
hp_param_spec = hp_config.get('parameter_spec', {})
hp_id_columns = hp_config.get('id_columns', [])
hp_eval_metric = hp_config.get('eval_metric', {'roc_auc': 'maximize'})
hp_use_package = hp_config.get('use_package_module', True)

# Encode HP tuning parameters
hp_parameter_spec_json = encode_json_arg(hp_param_spec)
hp_eval_metric_json = encode_json_arg(hp_eval_metric)

print("✅ Pipeline parameters prepared and encoded")
print(f"   Base64 encoding used for complex JSON structures")


✅ Pipeline parameters prepared and encoded
   Base64 encoding used for complex JSON structures


## Step 6: Create and Submit Pipeline Job


In [6]:
# Initialize Vertex AI
aiplatform.init(
    project=constants["PROJECT"],
    location=constants["LOCATION"],
)

# Determine pipeline name based on enabled components
if enable_fe and enable_hp:
    pipeline_func_name = "flexible-ml-pipeline-sequential"
    pipeline_desc = "Feature Engineering → Hyperparameter Tuning (Sequential)"
elif enable_fe:
    pipeline_func_name = "flexible-ml-pipeline-fe-only"
    pipeline_desc = "Feature Engineering Only"
elif enable_hp:
    pipeline_func_name = "flexible-ml-pipeline-hp-only"
    pipeline_desc = "Hyperparameter Tuning Only"
else:
    raise ValueError("At least one component (FE or HP) must be enabled!")

# Create pipeline job display name with timestamp
pipeline_display_name = f"{pipeline_func_name}-{datetime.now().strftime('%Y%m%d%H%M%S')}"

# Create pipeline job with all parameters
pipeline_job = aiplatform.PipelineJob(
    display_name=pipeline_display_name,
    template_path="flexible_ml_pipeline.json",
    pipeline_root=pipeline_root,
    enable_caching=False,
    encryption_spec_key_name=constants["CMEK_KEY"],
    parameter_values={
        # Project and infrastructure parameters
        "project_id": constants["PROJECT"],
        "region": constants["LOCATION"],
        "cmek_key": constants["CMEK_KEY"],
        "pipeline_root": pipeline_root,
        "package_uri": package_uri,
        "python_module": "trainer.task",
        "service_account": constants["SERVICE_ACCOUNT"],
        "docker_uri": constants["DOCKER_URI"],
        "lob": constants["LOB"],
        "shared_project": constants["SHARED_PROJECT"],
        "machine_type_param": config['vertex_ai'].get('machine_type', 'n1-standard-4'),
        "accelerator_type": accelerator_type_value,
        "accelerator_count": config['vertex_ai'].get('accelerator_count', 1),
        
        # GCP configuration
        "gcp_gcp_project": config['gcp']['gcp_project'],
        "gcp_db": config['gcp']['gcp_db'],
        "prefix": config['gcp']['prefix'],
        "default_exp": config['gcp']['default_exp'],
        "sdoh_year": config['gcp']['sdoh_year'],
        "location": config['gcp']['location'],
        "bucket_name": config['gcp']['bucket_name'],
        "gcs_destination_path": config['gcp']['gcs_destination_path'],
        
        # BigQuery labels
        "owner": config['bigquery_labels']['owner'],
        "costcenter": config['bigquery_labels']['costcenter'],
        "unique_id": config['bigquery_labels']['unique_id'],
        "pipeline_type": config['bigquery_labels']['pipeline_type'],
        "expiration_days": config['bigquery_labels']['expiration_days'],
        
        # Feature engineering parameters
        "sql_queries": sql_queries_json,
        "random_seed": config['data_processing']['random_seed'],
        "numpy_seed": config['data_processing']['numpy_seed'],
        "embedding_pattern": config['data_processing']['embedding_pattern'],
        "variance_threshold": config['data_processing']['variance_threshold'],
        "target_column": config['data_processing']['target_column'],
        "exclude_for_variance": exclude_for_variance_json,
        "exclude_for_classification": exclude_for_classification_json,
        "feature_classification": feature_classification_json,
        "min_occurrence": config['one_hot_encoding']['min_occurrence'],
        "model_config": model_config_json,
        "undersampling_config": undersampling_config_json,
        "rfecv_config": rfecv_config_json,
        "metrics_config": metrics_config_json,
        "output_config": output_config_json,
        "bigquery_query_config": bigquery_query_config_json,
        
        # Hyperparameter tuning parameters
        "hp_machine_type": hp_config.get('machine_type', 'n1-standard-8'),
        "hp_max_trials": hp_config.get('max_trial_count', 10),
        "hp_parallel_trials": hp_config.get('parallel_trial_count', 2),
        "hp_max_failed_trials": hp_config.get('max_failed_trial_count', 2),
        "hp_id_columns": encode_json_arg(hp_id_columns),
        "hp_parameter_spec": hp_parameter_spec_json,
        "hp_eval_metric": hp_eval_metric_json,
        "hp_use_package_module": hp_use_package,
        "hp_training_table": hp_config.get('training_table', '') or '',
        "hp_test_table": hp_config.get('test_table', '') or '',
        "hp_sql_queries": encode_json_arg(hp_config.get('sql_queries', {})),
        
        # Component enable flags
        "enable_feature_engineering": enable_fe,
        "enable_hyperparameter_tuning": enable_hp,
    },
    labels=constants["LABELS"]
)

print(f"\n✅ Pipeline job created successfully!")
print(f"   📋 Pipeline: {pipeline_desc}")
print(f"   🏷️  Name: {pipeline_display_name}")
print(f"   📦 Package: {package_uri}")
if enable_hp:
    print(f"   🔧 HP Tuning method: {'Package Module (trainer.hp_tuning.task)' if hp_use_package else 'GCS File (legacy)'}")
if enable_fe:
    output_config = config['output']
    print(f"\n   📊 Feature Engineering Output Tables:")
    print(f"      Training: {output_config['training_table_name']}")
    print(f"      Test: {output_config['test_table_name']}")
if enable_hp and not enable_fe:
    print(f"\n   📊 Hyperparameter Tuning Input Tables:")
    print(f"      Training: {hp_config.get('training_table', 'From config.yaml')}")
    print(f"      Test: {hp_config.get('test_table', 'From config.yaml')}")



✅ Pipeline job created successfully!
   📋 Pipeline: Hyperparameter Tuning Only
   🏷️  Name: flexible-ml-pipeline-hp-only-20260227193723
   📦 Package: gs://hcm-cm-de-code-hcb-dev/vertex-test/packages/feature_engineering_trainer-0.1.0.tar.gz
   🔧 HP Tuning method: Package Module (trainer.hp_tuning.task)

   📊 Hyperparameter Tuning Input Tables:
      Training: None
      Test: None


## Step 7: Run Pipeline

**Note:** Set `sync=False` to run asynchronously (recommended for long-running jobs). Set `sync=True` to wait for completion.


In [7]:
# Run the pipeline
print("🚀 Submitting pipeline job to Vertex AI...\n")

pipeline_job.run(
    service_account=constants["SERVICE_ACCOUNT"],
    sync=True  # Set to False to run asynchronously
)

print("\n✅ Pipeline execution completed!")


🚀 Submitting pipeline job to Vertex AI...

Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/flexible-ml-pipeline-20260227193728
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/flexible-ml-pipeline-20260227193728')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/flexible-ml-pipeline-20260227193728?project=46378383599
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/flexible-ml-pipeline-20260227193728 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/flexible-ml-pipeline-20260227193728 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/flexible-ml-pipeline-20260227193728 current state:
3
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/flexible-ml-pipeline-20260227193728 current state:
3
PipelineJ

KeyboardInterrupt: 

## 📝 Notes

### Sequential Execution:
- When both FE and HP are enabled, HP tuning will **wait** for FE to complete before starting
- HP tuning will automatically use the tables created by FE

### Table Reuse:
- To skip FE and reuse existing tables:
  1. Set `components.feature_engineering.enable: false` in config.yaml
  2. Set `hyperparameter_tuning.training_table` and `test_table` to your existing table names
  3. Set `components.hyperparameter_tuning.enable: true`

### Fixed Table Names:
- FE creates tables with names specified in `output.training_table_name` and `output.test_table_name`
- No timestamps are added, making tables easy to reference and reuse
- Tables will be overwritten on each FE run

### Monitoring:
- View pipeline progress in the [Vertex AI Pipelines Console](https://console.cloud.google.com/vertex-ai/pipelines)
- Check training logs in the Vertex AI Training section
- Output tables will be in BigQuery under the configured dataset
